In [ ]:
# 예제 1-1

from PyQt5.QtWidgets import *
import sys
import os


class BeepSound(QMainWindow):

    def __init__(self):
        super().__init__()

        # 윈도우 이름과 위치, 크기 지정
        self.setWindowTitle('삑 소리 내기')
        self.setGeometry(200, 200, 500, 100)

        # 버튼 생성
        shortBeepButton = QPushButton('짧게 삑', self)
        longBeepButton = QPushButton('길게 삑', self)
        quitButton = QPushButton('종료', self)

        # 레이블 생성
        self.label = QLabel('환영합니다!', self)

        # 버튼 위치와 크기 지정
        shortBeepButton.setGeometry(10, 10, 100, 30)
        longBeepButton.setGeometry(110, 10, 100, 30)
        quitButton.setGeometry(210, 10, 100, 30)

        # 레이블 위치와 크기 지정
        self.label.setGeometry(10, 40, 500, 70)

        # 버튼을 클릭했을 때 실행할 콜백 함수 지정
        shortBeepButton.clicked.connect(self.shortBeepFunction)
        longBeepButton.clicked.connect(self.longBeepFunction)
        quitButton.clicked.connect(self.quitFunction)


    # 짧은 삑 소리를 내는 함수
    def shortBeepFunction(self):
        self.label.setText('주파수 1000으로 0.5초 동안 삑 소리를 냅니다.')
        winsound.Beep(1000, 500)


    # 긴 삑 소리를 내는 함수
    def longBeepFunction(self):
        self.label.setText('주파수 1000으로 3초 동안 삑 소리를 냅니다.')
        winsound.Beep(1000, 3000)


    # 프로그램을 종료하는 함수
    def quitFunction(self):
        self.close()


# QApplication 객체 생성
app = QApplication(sys.argv)

# BeepSound 윈도우 객체 생성
win = BeepSound()

# 윈도우 화면에 표시
win.show()

# 이벤트 루프 실행
app.exec_()

0

In [ ]:
# 예제 1-2

import cv2 as cv
import numpy as np
import sys
from PyQt5.QtWidgets import *


class Orim(QMainWindow):

    def __init__(self):
        super().__init__()

        self.setWindowTitle('오림')
        self.setGeometry(200, 200, 700, 200)

        # 버튼 생성
        fileButton = QPushButton('파일', self)
        paintButton = QPushButton('페인팅', self)
        cutButton = QPushButton('오림', self)
        incButton = QPushButton('+', self)
        decButton = QPushButton('-', self)
        saveButton = QPushButton('저장', self)
        quitButton = QPushButton('나가기', self)

        # 버튼 위치와 크기 지정
        fileButton.setGeometry(10, 10, 100, 30)
        paintButton.setGeometry(110, 10, 100, 30)
        cutButton.setGeometry(210, 10, 100, 30)
        incButton.setGeometry(310, 10, 50, 30)
        decButton.setGeometry(360, 10, 50, 30)
        saveButton.setGeometry(410, 10, 100, 30)
        quitButton.setGeometry(510, 10, 100, 30)

        # 버튼 클릭과 함수 연결
        fileButton.clicked.connect(self.fileOpenFunction)
        paintButton.clicked.connect(self.paintFunction)
        cutButton.clicked.connect(self.cutFunction)
        incButton.clicked.connect(self.incFunction)
        decButton.clicked.connect(self.decFunction)
        saveButton.clicked.connect(self.saveFunction)
        quitButton.clicked.connect(self.quitFunction)

        # 페인팅 붓의 크기
        self.BrushSiz = 5

        # 파란색 물체, 빨간색 배경
        self.LColor, self.RColor = (255, 0, 0), (0, 0, 255)


    # 파일을 여는 함수
    def fileOpenFunction(self):
        fname = QFileDialog.getOpenFileName(self, 'Open file', './')

        self.img = cv.imread(fname[0])

        # 파일을 찾을 수 없는 경우 프로그램 종료
        if self.img is None:
            sys.exit('파일을 찾을 수 없습니다.')

        # 표시용 영상
        self.img_show = np.copy(self.img)

        cv.imshow('Painting', self.img_show)

        # 모든 화소를 배경일 것 같음으로 초기화
        self.mask = np.zeros(
            (self.img.shape[0], self.img.shape[1]),
            np.uint8
        )

        self.mask[:, :] = cv.GC_PR_BGD


    # 페인팅 버튼을 누르면 실행
    def paintFunction(self):
        cv.setMouseCallback('Painting', self.painting)


    # 마우스로 그림을 그리는 함수
    def painting(self, event, x, y, flags, param):

        if event == cv.EVENT_LBUTTONDOWN:
            cv.circle(
                self.img_show,
                (x, y),
                self.BrushSiz,
                self.LColor,
                -1
            )

            # 왼쪽 버튼을 클릭하면 파란색

            cv.circle(self.mask, (x, y), self.BrushSiz, cv.GC_FGD, -1)

        elif event == cv.EVENT_RBUTTONDOWN:
            cv.circle(self.img_show, (x, y), self.BrushSiz, self.RColor, -1)
        
            # 오른쪽 버튼을 클릭하면 빨간색

            cv.circle(self.mask, (x, y), self.BrushSiz, cv.GC_BGD, -1)

        elif event == cv.EVENT_MOUSEMOVE and flags == cv.EVENT_FLAG_LBUTTON:
            cv.circle(self.img_show, (x, y), self.BrushSiz, self.LColor, -1)
            
            # 왼쪽 버튼을 클릭하고 이동하면 파란색

            cv.circle(self.mask, (x, y), self.BrushSiz, cv.GC_FGD, -1)

        elif event == cv.EVENT_MOUSEMOVE and flags == cv.EVENT_FLAG_RBUTTON:
            cv.circle(self.img_show, (x, y), self.BrushSiz, self.RColor, -1)
            
            # 오른쪽 버튼을 클릭하고 이동하면 빨간색

            cv.circle(self.mask, (x, y), self.BrushSiz, cv.GC_BGD, -1)

        cv.imshow('Painting', self.img_show)


    # 오림 함수
    def cutFunction(self):

        # 배경과 전경의 모델을 저장할 배열 생성
        background = np.zeros((1, 65), np.float64)
        foreground = np.zeros((1, 65), np.float64)

        # GrabCut 알고리즘을 이용하여 영상에서 물체 추출
        cv.grabCut(
            self.img,
            self.mask,
            None,
            background,
            foreground,
            5,
            cv.GC_INIT_WITH_MASK
        )

        # 전경으로 판단되는 부분을 1, 나머지를 0으로 설정
        mask2 = np.where(
            (self.mask == 2) | (self.mask == 0),
            0,
            1
        ).astype('uint8')

        # 전경 부분만 추출
        self.grabImg = self.img * mask2[:, :, np.newaxis]

        # 오려낸 결과 영상 출력
        cv.imshow('Scissoring', self.grabImg)


    # 붓 크기를 증가시키는 함수
    def incFunction(self):
        self.BrushSiz = min(20, self.BrushSiz + 1)


    # 붓 크기를 감소시키는 함수
    def decFunction(self):
        self.BrushSiz = max(1, self.BrushSiz - 1)


    # 결과 영상을 저장하는 함수
    def saveFunction(self):
        fname = QFileDialog.getSaveFileName(self, '파일 저장', './')
        cv.imwrite(fname[0], self.grabImg)


    # 프로그램 종료 함수
    def quitFunction(self):
        cv.destroyAllWindows()
        self.close()


# QApplication 객체 생성
app = QApplication(sys.argv)

# Orim 윈도우 객체 생성
win = Orim()

# 윈도우 화면에 표시
win.show()

# 이벤트 루프 실행
app.exec_()

0

In [ ]:
# 예제 2

import cv2 as cv
import numpy as np
from PyQt5.QtWidgets import *
import sys
import os


class TrafficWeak(QMainWindow):

    def __init__(self):
        super().__init__()

        self.setWindowTitle('교통약자 보호')
        self.setGeometry(200, 200, 700, 200)

        # 버튼 생성
        signButton = QPushButton('표지판 등록', self)
        roadButton = QPushButton('도로 영상 불러옴', self)
        recognitionButton = QPushButton('인식', self)
        quitButton = QPushButton('나가기', self)

        # 레이블 생성
        self.label = QLabel('환영합니다!', self)

        # 버튼 위치와 크기 지정
        signButton.setGeometry(10, 10, 100, 30)
        roadButton.setGeometry(110, 10, 100, 30)
        recognitionButton.setGeometry(210, 10, 100, 30)
        quitButton.setGeometry(510, 10, 100, 30)

        # 레이블 위치와 크기 지정
        self.label.setGeometry(10, 40, 600, 170)

        # 버튼 클릭과 함수 연결
        signButton.clicked.connect(self.signFunction)
        roadButton.clicked.connect(self.roadFunction)
        recognitionButton.clicked.connect(self.recognitionFunction)
        quitButton.clicked.connect(self.quitFunction)

        # 표지판 모델 영상
        self.signFiles = [
            ['child.png', '어린이'],
            ['elder.png', '노인'],
            ['disabled.png', '장애인']
        ]

        # 표지판 모델 영상 저장
        self.signImgs = []


    # 표지판 등록 함수
    def signFunction(self):

        self.label.clear()
        self.label.setText('교통약자 표지판을 등록합니다.')

        # 등록된 표지판 영상을 하나씩 읽음
        for fname, _ in self.signFiles:
            self.signImgs.append(cv.imread(fname))
            cv.imshow(fname, self.signImgs[-1])


    # 도로 영상을 불러오는 함수
    def roadFunction(self):

        # 표지판이 등록되지 않은 경우
        if self.signImgs == []:
            self.label.setText('먼저 표지판을 등록하세요.')

        else:
            # 도로 영상 파일 선택
            fname = QFileDialog.getOpenFileName(
                self, '파일 읽기', './'
            )

            self.roadImg = cv.imread(fname[0])

            # 파일을 찾을 수 없는 경우
            if self.roadImg is None:
                sys.exit('파일을 찾을 수 없습니다.')

            # 도로 영상 출력
            cv.imshow('Road scene', self.roadImg)


    # 표지판 인식 함수
    def recognitionFunction(self):

        # 도로 영상이 없는 경우
        if self.roadImg is None:
            self.label.setText('먼저 도로 영상을 입력하세요.')

        else:

            # SIFT 객체 생성
            sift = cv.SIFT_create()

            # 여러 표지판 영상의 키포인트와 기술자 저장
            KD = []

            for img in self.signImgs:

                # 표지판 영상을 명암 영상으로 변환
                gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

                # 키포인트와 기술자 추출
                KD.append(sift.detectAndCompute(gray, None))


            # 도로 영상을 명암 영상으로 변환
            grayRoad = cv.cvtColor(
                self.roadImg,
                cv.COLOR_BGR2GRAY
            )

            # 키포인트와 기술자 추출
            road_kp, road_des = sift.detectAndCompute(
                grayRoad, None
            )


            # 특징점 매칭기 생성
            matcher = cv.DescriptorMatcher_create(
                cv.DescriptorMatcher_FLANNBASED
            )

            # 여러 표지판 영상의 good match를 저장
            GM = []

            # 각 표지판 영상에 대해 매칭
            for sign_kp, sign_des in KD:

                # KNN 매칭
                knn_match = matcher.knnMatch(
                    sign_des, road_des, 2
                )

                T = 0.7
                good_match = []

                # 가장 가까운 두 개의 특징점을 비교
                for nearest1, nearest2 in knn_match:

                    if (nearest1.distance / nearest2.distance) < T:
                        good_match.append(nearest1)

                GM.append(good_match)


            # 매칭 쌍 개수가 최대인 표지판 찾기
            best = GM.index(max(GM, key=len))


            # 최선의 표지판이 매칭 쌍 4개 미만이면 실패
            if len(GM[best]) < 4:
                self.label.setText('표지판이 없습니다.')

            else:

                # 가장 잘 매칭된 표지판의 키포인트
                sign_kp = KD[best][0]

                # 가장 잘 매칭된 good match
                good_match = GM[best]


                # 표지판 영상의 특징점 좌표
                points1 = np.float32(
                    [sign_kp[gm.queryIdx].pt
                     for gm in good_match]
                )

                # 도로 영상의 특징점 좌표
                points2 = np.float32(
                    [road_kp[gm.trainIdx].pt
                     for gm in good_match]
                )


                # Homography 계산
                H, _ = cv.findHomography(
                    points1,
                    points2,
                    cv.RANSAC
                )


                # 표지판 영상의 크기
                h1, w1 = self.signImgs[best].shape[0], \
                         self.signImgs[best].shape[1]

                # 도로 영상의 크기
                h2, w2 = self.roadImg.shape[0], \
                         self.roadImg.shape[1]


                # 표지판 영상의 네 꼭짓점 좌표
                box1 = np.float32([
                    [0, 0],
                    [0, h1 - 1],
                    [w1 - 1, h1 - 1],
                    [w1 - 1, 0]
                ]).reshape(4, 1, 2)


                # Homography를 이용하여 좌표 변환
                box2 = cv.perspectiveTransform(box1, H)


                # 도로 영상에 표지판 영역을 초록색으로 표시
                self.roadImg = cv.polylines(
                    self.roadImg,
                    [np.int32(box2)],
                    True,
                    (0, 255, 0),
                    4
                )


                # 매칭 결과를 보여주기 위한 빈 영상 생성
                img_match = np.empty(
                    (max(h1, h2), w1 + w2, 3),
                    dtype=np.uint8
                )


                # 특징점 매칭 결과 표시
                cv.drawMatches(
                    self.signImgs[best],
                    sign_kp,
                    self.roadImg,
                    road_kp,
                    good_match,
                    img_match,
                    flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
                )


                # 매칭 결과 영상 출력
                cv.imshow(
                    'Matches and Homography',
                    img_match
                )


                # 인식 결과 출력
                self.label.setText(
                    self.signFiles[best][1]
                    + ' 보호구역입니다. 30km로 서행하세요.'
                )

                # 경고음 발생
                winsound.Beep(3000, 500)


    # 프로그램 종료 함수
    def quitFunction(self):

        cv.destroyAllWindows()
        self.close()


# QApplication 객체 생성
app = QApplication(sys.argv)

# TrafficWeak 객체 생성
win = TrafficWeak()

# 윈도우 화면에 표시
win.show()

# 이벤트 루프 실행
app.exec_()

0

In [ ]:
# 예제 3

import cv2 as cv
import numpy as np
from PyQt5.QtWidgets import *
import sys


class SpecialEffect(QMainWindow):

    def __init__(self):
        super().__init__()

        self.setWindowTitle('사진 특수 효과')
        self.setGeometry(200, 200, 800, 200)

        # 버튼 생성
        pictureButton = QPushButton('사진 읽기', self)
        embossButton = QPushButton('엠보싱', self)
        cartoonButton = QPushButton('카툰', self)
        sketchButton = QPushButton('연필 스케치', self)
        oilButton = QPushButton('유화', self)
        saveButton = QPushButton('저장하기', self)

        # 특수 효과 선택 콤보박스
        self.pickCombo = QComboBox(self)

        self.pickCombo.addItems([
            '엠보싱',
            '카툰',
            '연필 스케치(명암)',
            '연필 스케치(컬러)',
            '유화'
        ])

        # 종료 버튼
        quitButton = QPushButton('나가기', self)

        # 레이블 생성
        self.label = QLabel('환영합니다!', self)

        # 버튼 위치와 크기 지정
        pictureButton.setGeometry(10, 10, 100, 30)
        embossButton.setGeometry(110, 10, 100, 30)
        cartoonButton.setGeometry(210, 10, 100, 30)
        sketchButton.setGeometry(310, 10, 100, 30)
        oilButton.setGeometry(410, 10, 100, 30)
        self.pickCombo.setGeometry(510, 10, 110, 30)
        quitButton.setGeometry(620, 10, 100, 30)

        saveButton.setGeometry(510, 50, 100, 30)

        self.label.setGeometry(10, 40, 500, 170)

        # 버튼 클릭과 함수 연결
        pictureButton.clicked.connect(self.pictureOpenFunction)
        embossButton.clicked.connect(self.embossFunction)
        cartoonButton.clicked.connect(self.cartoonFunction)
        sketchButton.clicked.connect(self.sketchFunction)
        oilButton.clicked.connect(self.oilFunction)
        saveButton.clicked.connect(self.saveFunction)
        quitButton.clicked.connect(self.quitFunction)


    # 사진을 불러오는 함수
    def pictureOpenFunction(self):

        fname = QFileDialog.getOpenFileName(
            self, '사진 읽기', './'
        )

        self.img = cv.imread(fname[0])

        # 사진을 찾을 수 없는 경우
        if self.img is None:
            sys.exit('파일을 찾을 수 없습니다.')

        # 원본 사진 출력
        cv.imshow('Painting', self.img)


    # 엠보싱 효과 함수
    def embossFunction(self):

        # 엠보싱 필터
        femboss = np.array([
            [-1.0, 0.0, 0.0],
            [ 0.0, 0.0, 0.0],
            [ 0.0, 0.0, 1.0]
        ])

        # 컬러 영상을 명암 영상으로 변환
        gray = cv.cvtColor(
            self.img,
            cv.COLOR_BGR2GRAY
        )

        # 명암 영상을 16비트 정수형으로 변환
        gray16 = np.int16(gray)

        # 엠보싱 필터 적용
        self.emboss = np.uint8(
            np.clip(
                cv.filter2D(gray16, -1, femboss) + 128,
                0,
                255
            )
        )

        # 결과 영상 출력
        cv.imshow('Emboss', self.emboss)


    # 카툰 효과 함수
    def cartoonFunction(self):

        self.cartoon = cv.stylization(
            self.img,
            sigma_s=60,
            sigma_r=0.45
        )

        # 카툰 영상 출력
        cv.imshow('Cartoon', self.cartoon)


    # 연필 스케치 효과 함수
    def sketchFunction(self):

        self.sketch_gray, self.sketch_color = cv.pencilSketch(
            self.img,
            sigma_s=60,
            sigma_r=0.07,
            shade_factor=0.02
        )

        # 명암 연필 스케치 출력
        cv.imshow(
            'Pencil sketch(gray)',
            self.sketch_gray
        )

        # 컬러 연필 스케치 출력
        cv.imshow(
            'Pencil sketch(color)',
            self.sketch_color
        )


    # 유화 효과 함수
    def oilFunction(self):

        self.oil = cv.xphoto.oilPainting(
            self.img,
            10,
            1,
            cv.COLOR_BGR2Lab
        )

        # 유화 영상 출력
        cv.imshow('Oil painting', self.oil)


    # 영상 저장 함수
    def saveFunction(self):

        fname = QFileDialog.getSaveFileName(
            self,
            '파일 저장',
            './'
        )

        # 현재 선택된 효과의 번호
        i = self.pickCombo.currentIndex()

        # 엠보싱 영상 저장
        if i == 0:
            cv.imwrite(fname[0], self.emboss)

        # 카툰 영상 저장
        elif i == 1:
            cv.imwrite(fname[0], self.cartoon)

        # 명암 연필 스케치 저장
        elif i == 2:
            cv.imwrite(fname[0], self.sketch_gray)

        # 컬러 연필 스케치 저장
        elif i == 3:
            cv.imwrite(fname[0], self.sketch_color)

        # 유화 영상 저장
        elif i == 4:
            cv.imwrite(fname[0], self.oil)


    # 프로그램 종료 함수
    def quitFunction(self):

        cv.destroyAllWindows()
        self.close()


# QApplication 객체 생성
app = QApplication(sys.argv)

# SpecialEffect 객체 생성
win = SpecialEffect()

# 윈도우 화면에 표시
win.show()

# 이벤트 루프 실행
app.exec_()

0

In [1]:
# 1006 과제

import cv2 as cv
import numpy as np
from PyQt5.QtWidgets import *
import sys


class SpecialEffect(QMainWindow):

    def __init__(self):
        super().__init__()

        self.setWindowTitle('사진 특수 효과')
        self.setGeometry(200, 200, 850, 250)

        # 버튼 생성
        pictureButton = QPushButton('사진 읽기', self)
        embossButton = QPushButton('엠보싱', self)
        cartoonButton = QPushButton('카툰', self)
        sketchButton = QPushButton('연필 스케치', self)
        oilButton = QPushButton('유화', self)
        selectButton = QPushButton('선택', self)
        saveButton = QPushButton('저장하기', self)
        quitButton = QPushButton('나가기', self)

        # 콤보박스 생성
        self.pickCombo = QComboBox(self)

        self.pickCombo.addItems([
            '엠보싱',
            '카툰',
            '연필 스케치(명암)',
            '연필 스케치(컬러)',
            '유화'
        ])

        # 레이블 생성
        self.label = QLabel('환영합니다!', self)

        # 버튼 위치와 크기 지정
        pictureButton.setGeometry(10, 10, 100, 30)
        embossButton.setGeometry(110, 10, 100, 30)
        cartoonButton.setGeometry(210, 10, 100, 30)
        sketchButton.setGeometry(310, 10, 100, 30)
        oilButton.setGeometry(410, 10, 100, 30)

        self.pickCombo.setGeometry(510, 10, 120, 30)

        selectButton.setGeometry(510, 50, 100, 30)
        saveButton.setGeometry(620, 50, 100, 30)
        quitButton.setGeometry(730, 50, 100, 30)

        self.label.setGeometry(10, 90, 600, 100)

        # 버튼 클릭과 함수 연결
        pictureButton.clicked.connect(self.pictureOpenFunction)
        embossButton.clicked.connect(self.embossFunction)
        cartoonButton.clicked.connect(self.cartoonFunction)
        sketchButton.clicked.connect(self.sketchFunction)
        oilButton.clicked.connect(self.oilFunction)

        # 과제에서 추가한 선택 버튼
        selectButton.clicked.connect(self.selectFunction)

        saveButton.clicked.connect(self.saveFunction)
        quitButton.clicked.connect(self.quitFunction)


    # 사진을 읽는 함수
    def pictureOpenFunction(self):

        fname = QFileDialog.getOpenFileName(
            self, '사진 읽기', './'
        )

        self.img = cv.imread(fname[0])

        if self.img is None:
            sys.exit('파일을 찾을 수 없습니다.')

        cv.imshow('Painting', self.img)


    # 엠보싱 효과
    def embossFunction(self):

        femboss = np.array([
            [-1.0, 0.0, 0.0],
            [0.0, 0.0, 0.0],
            [0.0, 0.0, 1.0]
        ])

        # 컬러 영상을 명암 영상으로 변환
        gray = cv.cvtColor(
            self.img,
            cv.COLOR_BGR2GRAY
        )

        gray16 = np.int16(gray)

        # 엠보싱 필터 적용
        self.emboss = np.uint8(
            np.clip(
                cv.filter2D(gray16, -1, femboss) + 128,
                0,
                255
            )
        )

        cv.imshow('Emboss', self.emboss)


    # 카툰 효과
    def cartoonFunction(self):

        self.cartoon = cv.stylization(
            self.img,
            sigma_s=60,
            sigma_r=0.45
        )

        cv.imshow('Cartoon', self.cartoon)


    # 연필 스케치 효과
    def sketchFunction(self):

        self.sketch_gray, self.sketch_color = cv.pencilSketch(
            self.img,
            sigma_s=60,
            sigma_r=0.07,
            shade_factor=0.02
        )

        cv.imshow(
            'Pencil sketch(gray)',
            self.sketch_gray
        )

        cv.imshow(
            'Pencil sketch(color)',
            self.sketch_color
        )


    # 유화 효과
    def oilFunction(self):

        self.oil = cv.xphoto.oilPainting(
            self.img,
            10,
            1,
            cv.COLOR_BGR2Lab
        )

        cv.imshow('Oil painting', self.oil)


    # ⭐ 과제 추가 기능
    # 콤보박스에서 선택한 효과를 실행
    def selectFunction(self):

        i = self.pickCombo.currentIndex()

        if i == 0:
            self.embossFunction()

        elif i == 1:
            self.cartoonFunction()

        elif i == 2:
            self.sketchFunction()
            cv.destroyWindow('Pencil sketch(color)')

        elif i == 3:
            self.sketchFunction()
            cv.destroyWindow('Pencil sketch(gray)')

        elif i == 4:
            self.oilFunction()


    # 선택한 효과를 파일로 저장
    def saveFunction(self):

        fname = QFileDialog.getSaveFileName(
            self,
            '파일 저장',
            './'
        )

        i = self.pickCombo.currentIndex()

        if i == 0:
            cv.imwrite(fname[0], self.emboss)

        elif i == 1:
            cv.imwrite(fname[0], self.cartoon)

        elif i == 2:
            cv.imwrite(fname[0], self.sketch_gray)

        elif i == 3:
            cv.imwrite(fname[0], self.sketch_color)

        elif i == 4:
            cv.imwrite(fname[0], self.oil)


    # 프로그램 종료
    def quitFunction(self):

        cv.destroyAllWindows()
        self.close()


# 프로그램 실행
app = QApplication(sys.argv)

win = SpecialEffect()
win.show()

app.exec_()

AttributeError: 'SpecialEffect' object has no attribute 'img'

0